# Guided Lab: MCP Core Concepts and a LangChain Agent — Bramblewood Gear Co. / FieldWise

**Use case:** Bramblewood wants FieldWise's retrieval capability exposed as
a standard MCP server, so the same search tool can plug into several
internal apps (the support console, a Slack ops bot, and so on) instead of
each one reimplementing its own retrieval code.

What you will build:
- an MCP server with `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()`
- a stdio transport server for local development
- a LangChain agent that loads MCP tools, resources, and prompts
- a structured-output schema for the final answer
- a practical support-ops workflow backed by Bramblewood's own handbook

MCP standardizes how hosts, clients, and servers exchange tools and
context. LangChain's MCP docs describe agents using tools defined on MCP
servers through `langchain-mcp-adapters`, and the MCP architecture docs
describe the host/client/server split. The MCP SDK's `FastMCP` exposes
tool, resource, and prompt decorators, and MCP Inspector is the standard
tool for debugging a server directly.

## Core concepts

- **Host**: the app users interact with, such as this notebook, a Slack
  bot, or Bramblewood's support console.
- **Client**: the protocol component inside the host that connects to MCP
  servers.
- **Server**: the program exposing tools, resources, and prompts -- here,
  FieldWise's handbook search.
- **Transport**: the channel used for communication, such as `stdio` or an
  HTTP-based transport.

`stdio` is the simplest transport for local development. For a server
running elsewhere, HTTP-based transports carry JSON-RPC over POST, with
`streamable_http` as the current standard and `sse` kept around mostly for
compatibility with older clients.

## Learning goals

By the end of this notebook, you should be able to:

1. Explain what MCP solves.
2. Build a small MCP server around a real retrieval index.
3. Expose a tool, resource, and prompt.
4. Connect the server to a LangChain agent.
5. Understand schema validation for tool inputs and the final structured
   output.
6. Inspect the server with MCP Inspector.

## 1) Install packages

In [ ]:
%pip install -qU python-dotenv reportlab pypdf scikit-learn langchain langchain-core langchain-groq langchain-mcp-adapters langchain-chroma chromadb mcp

## 2) Load environment variables

Create a `.env` file like this:

```env
GROQ_API_KEY=your_groq_api_key
GROQ_MODEL=llama-3.3-70b-versatile
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=fieldwise-mcp-support-lab
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("GROQ_MODEL:", GROQ_MODEL)
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))

if not LANGSMITH_API_KEY:
    os.environ["LANGSMITH_TRACING"] = "false"

## 3) Build the handbook PDF

The same *Product Care & Warranty Handbook* used in the end-to-end pipeline
notebook -- generated here so this notebook stays runnable on its own.

In [ ]:
from pathlib import Path
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

DATA_DIR = Path("./fieldwise_mcp_data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "bramblewood_handbook.pdf"


def build_handbook_pdf(path: Path) -> Path:
    if path.exists() and path.stat().st_size > 0:
        print(f"Using existing handbook: {path}")
        return path

    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(str(path), pagesize=LETTER, topMargin=0.9 * inch, bottomMargin=0.9 * inch)
    story = []

    def heading(text):
        story.append(Paragraph(text, styles["Heading1"]))
        story.append(Spacer(1, 8))

    def body(text):
        story.append(Paragraph(text, styles["BodyText"]))
        story.append(Spacer(1, 10))

    story.append(Paragraph("Bramblewood Gear Co.", styles["Title"]))
    story.append(Paragraph("Product Care &amp; Warranty Handbook", styles["Title"]))
    story.append(Spacer(1, 20))
    body(
        "This handbook collects gear-care instructions, troubleshooting steps, "
        "and warranty terms for Bramblewood Gear Co. products, and is the "
        "source document FieldWise's MCP server searches over."
    )
    story.append(PageBreak())

    heading("1. Tents")
    body(
        "Summit Ridge 2P Tent: a 3-season, freestanding tent with aluminum "
        "poles and a silicone-coated rainfly."
    )
    body(
        "Alpine Trail 4P Tent: a 4-season expedition tent with a "
        "HydroShield-coated rainfly and reinforced poles rated for high "
        "wind and snow load."
    )
    body(
        "Both tents ship with a factory-applied waterproof coating on the "
        "rainfly that gradually wears down with UV exposure, which can let "
        "water seep through the seams during sustained rain. Reapply a DWR "
        "(durable water repellent) treatment to the rainfly once a year and "
        "reseal the seams with seam sealer. If leaking continues after "
        "reproofing and the tent is less than two years old, treat it as a "
        "hard-goods warranty claim rather than a general return."
    )
    story.append(PageBreak())

    heading("2. Sleeping Bags")
    body(
        "Ember Down 20F Sleeping Bag: 650-fill DriDown, a water-resistant "
        "treated goose down rated to 20 degrees Fahrenheit. Wash only with "
        "technical down soap on a gentle cycle and tumble dry low with "
        "clean tennis balls to restore loft; regular detergent strips the "
        "DriDown treatment and causes clumping."
    )
    story.append(PageBreak())

    heading("3. Backpacks")
    body(
        "Basecamp 65L Backpack: an internal-frame pack with an adjustable "
        "torso length. The main compartment zipper slider is the most "
        "common point of failure after heavy use; a stiff slider can "
        "usually be freed with dry silicone lubricant, and a fully broken "
        "slider is covered under Basecamp's lifetime workmanship warranty."
    )
    story.append(PageBreak())

    heading("4. Stoves")
    body(
        "TrailFlame Compact Stove: a canister-fed stove with piezo "
        "ignition. A piezo igniter that stops sparking is not a safety "
        "issue -- any lighter or match can ignite the burner -- and an "
        "igniter that fails within two years is covered under the "
        "standard hard-goods warranty."
    )
    story.append(PageBreak())

    heading("5. Footwear")
    body(
        "Ridgeline Hiking Boots: a waterproof membrane under leather and "
        "mesh uppers, with a resoleable Vibram outsole. An annual DWR "
        "reproofing spray restores the waterproof membrane; a paid resole "
        "program covers the outsole separately from the one-year "
        "manufacturing warranty."
    )
    story.append(PageBreak())

    heading("6. Returns &amp; Warranty Policy")
    body(
        "Unused gear in original packaging can be returned within 60 days "
        "for a full refund. Used or worn gear is handled through a "
        "warranty claim instead of a general return: 2 years on tents and "
        "stoves, lifetime on backpack stitching, and 1 year on footwear "
        "manufacturing defects."
    )

    doc.build(story)
    print(f"Built: {path}")
    return path


build_handbook_pdf(PDF_PATH)

## 4) Load and chunk the PDF

LangChain's knowledge-base flow is load -> split -> embed -> store. We use
that pattern here so the MCP tool has real document content to search.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))
print("First page preview:")
print(pages[0].page_content[:400])

## 5) Build a local Chroma index

The Chroma index powers the MCP search tool. We check once for a live
embedding model and fall back to a deterministic offline one if it isn't
reachable -- and importantly, the MCP server (written to disk in Section 8,
running as its own subprocess) needs to make the *same* choice, since a
persisted index has to be searched with an embedding function that
produces vectors in the same space it was built with.

In [ ]:
import shutil
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from sklearn.feature_extraction.text import HashingVectorizer


class OfflineHashingEmbeddings(Embeddings):
    """Deterministic, dependency-free embedding fallback. Uses the hashing
    trick, so it needs no fitted vocabulary and produces the same vector
    for the same text regardless of which process calls it -- this is what
    lets the MCP server subprocess reopen this notebook's Chroma index
    safely, without sharing any state with it."""

    def __init__(self, n_features: int = 512):
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        print("Embeddings: live sentence-transformers model loaded (semantic).")
        return candidate, "live"
    except Exception as e:
        print(f"Embeddings: falling back to offline hashing embeddings -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineHashingEmbeddings(), "offline"


embeddings, EMBEDDINGS_MODE = get_embeddings()

CHROMA_DIR = DATA_DIR / "chroma_store"
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="bramblewood_handbook_mcp",
    persist_directory=str(CHROMA_DIR),
)

print("Chroma store ready at", CHROMA_DIR.resolve(), "| embeddings mode:", EMBEDDINGS_MODE)

## 6) Create the local SQLite store

SQLite stores structured data: the product catalog, saved support notes,
and a log of what agents have searched for.

In [ ]:
import sqlite3
from datetime import datetime

SQLITE_DB = DATA_DIR / "fieldwise_support.db"
if SQLITE_DB.exists():
    SQLITE_DB.unlink()

conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

cur.execute("""
CREATE TABLE product_catalog (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    warranty_terms TEXT NOT NULL,
    care_summary TEXT NOT NULL
)
""")
cur.execute("""
CREATE TABLE support_notes (
    note_id INTEGER PRIMARY KEY AUTOINCREMENT,
    page_number INTEGER NOT NULL,
    note_text TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")
cur.execute("""
CREATE TABLE query_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    intent TEXT NOT NULL,
    source TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

catalog_rows = [
    ("Summit Ridge 2P Tent", "tents", "2-year manufacturing defect warranty", "Annual DWR reproofing and seam sealing."),
    ("Alpine Trail 4P Tent", "tents", "2-year manufacturing defect warranty", "Annual DWR reproofing; inspect poles after high-wind trips."),
    ("Ember Down 20F Sleeping Bag", "sleeping_bags", "2-year manufacturing defect warranty", "Wash with technical down soap only; tumble dry low with tennis balls."),
    ("Basecamp 65L Backpack", "backpacks", "Lifetime workmanship warranty", "Silicone lubricant for stiff zippers; free repair for broken sliders."),
    ("TrailFlame Compact Stove", "stoves", "2-year manufacturing defect warranty", "Use a backup lighter if the piezo igniter fails."),
    ("Ridgeline Hiking Boots", "footwear", "1-year manufacturing defect warranty", "Annual DWR spray; resole program available for a flat fee."),
]
cur.executemany(
    "INSERT INTO product_catalog (name, category, warranty_terms, care_summary) VALUES (?, ?, ?, ?)",
    catalog_rows,
)

for chunk in chunks[:8]:
    cur.execute(
        "INSERT INTO support_notes (page_number, note_text, created_at) VALUES (?, ?, ?)",
        (
            int(chunk.metadata.get("page", 0)),
            chunk.page_content[:600].replace("\n", " "),
            datetime.utcnow().isoformat(),
        ),
    )

conn.commit()
conn.close()

print("SQLite database created:", SQLITE_DB.resolve())

## 7) Inspect the structured data

In [ ]:
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

print("product_catalog rows:")
for row in cur.execute("SELECT name, category, warranty_terms FROM product_catalog"):
    print(" ", row)

print("\nsupport_notes sample rows:")
for row in cur.execute("SELECT note_id, page_number, substr(note_text, 1, 90) FROM support_notes LIMIT 3"):
    print(" ", row)

conn.close()

## 8) Write the MCP server

`FastMCP` exposes a tool, resource, and prompt from plain Python functions;
type hints and docstrings generate the tool's schema automatically, which
is what gives you input validation for free. This server re-opens the same
Chroma collection and SQLite database built above, so it needs its own copy
of the offline embedding fallback -- it runs as a separate process and
can't import anything from this notebook's kernel.

In [ ]:
%%writefile fieldwise_mcp_server.py
from pathlib import Path
import sqlite3

from mcp.server.fastmcp import FastMCP
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma

BASE_DIR = Path(".")
CHROMA_DIR = BASE_DIR / "fieldwise_mcp_data" / "chroma_store"
SQLITE_DB = BASE_DIR / "fieldwise_mcp_data" / "fieldwise_support.db"

mcp = FastMCP("fieldwise-support", json_response=True)


class OfflineHashingEmbeddings(Embeddings):
    "Deterministic, dependency-free embedding fallback (see the main notebook for the rationale)."

    def __init__(self, n_features: int = 512):
        from sklearn.feature_extraction.text import HashingVectorizer
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        return candidate
    except Exception:
        return OfflineHashingEmbeddings()


EMBEDDINGS = get_embeddings()
STORE = Chroma(
    collection_name="bramblewood_handbook_mcp",
    persist_directory=str(CHROMA_DIR),
    embedding_function=EMBEDDINGS,
)


def _product_catalog_summary() -> str:
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()
    rows = cur.execute("SELECT name, category, warranty_terms FROM product_catalog").fetchall()
    conn.close()
    if not rows:
        return "No product catalog available."
    return "\n".join(f"{name} ({category}): {warranty}" for name, category, warranty in rows)


@mcp.tool()
def search_handbook(query: str, k: int = 3) -> str:
    "Search the Bramblewood gear-care and warranty handbook for relevant passages."
    docs = STORE.similarity_search(query, k=k)
    lines = []
    for doc in docs:
        page = doc.metadata.get("page", "n/a")
        snippet = doc.page_content[:700].replace("\n", " ")
        lines.append(f"[page={page}] {snippet}")
    return "\n\n".join(lines)


@mcp.resource("handbook://catalog")
def catalog_resource() -> str:
    "Expose the product catalog as a readable resource."
    return _product_catalog_summary()


@mcp.resource("handbook://note/{note_id}")
def note_resource(note_id: str) -> str:
    "Expose a saved support note from SQLite."
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()
    row = cur.execute(
        "SELECT page_number, note_text FROM support_notes WHERE note_id = ?", (note_id,)
    ).fetchone()
    conn.close()
    if not row:
        return f"No note found for id={note_id}"
    return f"Page {row[0]}: {row[1]}"


@mcp.prompt()
def support_reply(topic: str, audience: str = "support agent") -> str:
    "Return a reusable support-reply prompt template."
    return (
        f"Write a short reply for a {audience}.\n\n"
        f"Topic: {topic}\n\n"
        "Use only the evidence from the handbook. "
        "Include: Summary, Steps, Warranty status, and Recommended action."
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 9) MCP transports

Use **stdio** for local development, which is what this lab uses: the
client launches `fieldwise_mcp_server.py` as a subprocess and talks to it
over stdin/stdout, so nothing here depends on network access. For a server
running elsewhere, `streamable_http` is the current standard HTTP-based
transport, with `sse` kept around mostly for compatibility with older
clients.

In [ ]:
# stdio (local, recommended for this lab)
stdio_config = {
    "fieldwise": {
        "transport": "stdio",
        "command": "python3",
        "args": ["fieldwise_mcp_server.py"],
    }
}

# streamable_http example, for a server running elsewhere (not used in this lab)
# http_config = {
#     "fieldwise": {
#         "transport": "streamable_http",
#         "url": "http://localhost:8000/mcp",
#     }
# }

stdio_config

## 10) MCP Inspector

MCP Inspector is the first-stop debugging tool for an MCP server: it can
connect over stdio or Streamable HTTP and lets you invoke tools, prompts,
and resources interactively, outside of any agent framework.

In [ ]:
print("To inspect the stdio server:")
print("npx -y @modelcontextprotocol/inspector python3 fieldwise_mcp_server.py")
print()
print("To inspect a remote HTTP server:")
print("npx -y @modelcontextprotocol/inspector http://localhost:8000/mcp")

## 11) Connect MCP tools, resources, and prompts with LangChain

`MultiServerMCPClient` loads tools, resources, and prompts from one or more
MCP servers. This step is pure local process communication over stdio --
no network access required, so it runs the same way in any environment.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(stdio_config)

tools = await client.get_tools()
blobs = await client.get_resources("fieldwise", uris=["handbook://catalog"])
prompt_messages = await client.get_prompt(
    "fieldwise",
    "support_reply",
    arguments={"topic": "tent rainfly leaking after 8 months", "audience": "support agent"},
)

print("Loaded MCP tools:", [t.name for t in tools])
print("\nLoaded resource preview:")
for blob in blobs:
    print(blob.as_string())
print("\nLoaded prompt messages:")
for message in prompt_messages:
    print(message.type, ":", message.content)

## 12) Groq availability check

The rest of this lab builds a LangChain agent on top of the MCP tools,
which needs a real tool-calling chat model. We check once whether Groq is
reachable. If it is, Section 13 builds a real `create_agent` agent exactly
as you'd run it live. If it isn't, Section 14 falls back to calling the
same MCP tool directly and building the structured response with a small
template instead of a live model -- the MCP wiring above (server, tool,
resource, prompt) is identical either way; only the final reasoning step
changes.

In [ ]:
def check_groq():
    try:
        from langchain_groq import ChatGroq
        candidate = ChatGroq(model=GROQ_MODEL, temperature=0)
        candidate.invoke("connectivity check")
        print("Groq: live connection confirmed.")
        return candidate, "live"
    except Exception as e:
        print(f"Groq: not reachable -- {type(e).__name__}: {str(e)[:160]}. Using the offline equivalent in Section 14 instead of a live agent.")
        return None, "offline"


groq_llm, GROQ_MODE = check_groq()

## 13) Build the LangChain agent on top of MCP tools (live path)

`create_agent` is LangChain's standard agent harness, running on
LangGraph, so the MCP tools become ordinary agent tools with no extra
wiring. This cell only runs the live agent if Groq is reachable; otherwise
it's skipped in favor of Section 14's offline equivalent.

In [ ]:
from pydantic import BaseModel, Field


class SupportResolution(BaseModel):
    summary: str = Field(description="One or two sentence summary of the issue and the resolution")
    steps: list[str] = Field(description="Concrete steps to relay to the customer or apply next")
    warranty_eligible: bool = Field(description="Whether this looks covered under warranty based on the handbook")
    recommended_action: str = Field(description="The single most important next action")


agent = None
if GROQ_MODE == "live":
    from langchain.agents import create_agent

    agent = create_agent(
        groq_llm,
        tools,
        system_prompt=(
            "You are FieldWise, Bramblewood's support-ops assistant. Use the "
            "search_handbook tool to retrieve evidence and ground your answer "
            "in the handbook. Keep it concise."
        ),
        response_format=SupportResolution,
    )
    print("Live agent ready.")
else:
    print("Skipping live agent build -- Groq is not reachable in this run.")

## 14) Run a grounded question

Live path: the agent decides to call `search_handbook`, then returns a
validated `SupportResolution`. Offline path: we call the exact same MCP
tool directly and build the same shape of structured response with a small
template instead of a live model, so the notebook still runs end-to-end
and still exercises the real MCP server.

In [ ]:
import re

QUESTION = "A customer says their Summit Ridge tent rainfly is leaking after 8 months of use. What should I tell them, and is this covered under warranty?"

if GROQ_MODE == "live":
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": QUESTION}]
    })
else:
    raw_result = await tools[0].ainvoke({"query": QUESTION, "k": 3})
    # A LangChain-wrapped MCP tool call returns a list of content blocks
    # (each a dict with a "type" and "text"), not a plain string.
    if isinstance(raw_result, list):
        tool_result = "\n".join(
            block.get("text", "") for block in raw_result if isinstance(block, dict)
        )
    else:
        tool_result = str(raw_result)
    print("Raw search_handbook tool result:\n", tool_result, "\n")

    sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", tool_result)
        if len(s.strip()) > 15
    ]
    warranty_eligible = "warranty" in tool_result.lower() and "2 year" in tool_result.lower().replace("-", " ")
    result = {
        "structured_response": SupportResolution(
            summary=f"[offline equivalent] Handbook guidance for: {QUESTION}",
            steps=sentences[:3] or ["Escalate to a human agent for review."],
            warranty_eligible=warranty_eligible,
            recommended_action=sentences[0] if sentences else "Escalate to a human agent for review.",
        )
    }

result

## 15) Inspect the structured response

In [ ]:
structured = result["structured_response"]
structured

## 16) Why this validates schemas well

On the server side, MCP tool schemas are generated from Python type hints
and docstrings in `FastMCP`. On the agent side, LangChain's structured
output validates the final answer against `SupportResolution` and raises a
clear error if the model's output doesn't match -- so a support console
consuming this agent's output can rely on `warranty_eligible` always being
a real boolean, `steps` always being a list, and so on.

## 17) Example of a direct resource and prompt workflow

Resources are read-only context; prompts are reusable templates the client
loads and feeds into a workflow. LangChain converts MCP prompts into
messages, which is why `prompt_messages` from Section 11 is already a list
of chat messages ready to use.

In [ ]:
for message in prompt_messages:
    print(message.type, ":", message.content)

## 18) What this notebook demonstrates

- MCP solves the host/client/server integration problem for LLM apps.
- `stdio` is the simplest transport for local development, and needs no
  network access -- the server, tool loading, resource loading, and prompt
  loading in this notebook all ran as real local process communication,
  regardless of whether Groq was reachable.
- `FastMCP` gives you `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()`
  from plain Python functions.
- LangChain's MCP adapters let an agent consume those server capabilities
  directly, with structured output validation on the way out.

## Key takeaways

The practical pattern: keep the source document in a local semantic store,
expose it through MCP as tools/resources/prompts, and let a LangChain agent
(or, offline, a direct tool call plus a template) consume those
capabilities. That gives Bramblewood a clean separation between the host
app, the client connection, the MCP server, and the transport layer --
several internal tools can share one `search_handbook` implementation
instead of each reimplementing retrieval.

## References

- MCP architecture overview: https://modelcontextprotocol.io/docs/learn/architecture
- MCP clients: https://modelcontextprotocol.io/docs/learn/client-concepts
- MCP servers: https://modelcontextprotocol.io/docs/learn/server-concepts
- MCP Inspector: https://modelcontextprotocol.io/docs/tools/inspector
- Build an MCP server: https://modelcontextprotocol.io/docs/develop/build-server
- LangChain MCP integration: https://docs.langchain.com/oss/python/langchain/mcp
- LangChain MCP tools: https://docs.langchain.com/oss/python/deepagents/code/mcp-tools

## Verification summary

| Section | Status | Notes |
|---|---|---|
| Package install | Executed | pip install cell |
| Handbook PDF generation + load + split | Executed & verified | real PDF on disk, 7 pages |
| Embeddings + Chroma index | Executed & verified | see printed `EMBEDDINGS_MODE` |
| SQLite catalog / notes / query log | Executed & verified | real local database |
| MCP server (tool, resource, prompt) | Executed & verified | real local subprocess over stdio, no network required |
| `MultiServerMCPClient` tool/resource/prompt loading | Executed & verified | real local IPC |
| Groq availability check | Executed & verified | see printed `GROQ_MODE` |
| Live `create_agent` agent + structured output | Written, executed only when `GROQ_MODE == "live"` | needs a real Groq connection; falls back to Section 14's offline equivalent otherwise |
| Offline equivalent (direct tool call + template) | Executed & verified when `GROQ_MODE == "offline"` | exercises the real MCP tool, not a simulation of it |

On a machine with a working `GROQ_API_KEY` and network access, `GROQ_MODE`
will read `live`, Section 13 builds a real agent, and Section 14 runs it
instead of the offline equivalent -- everything else in this notebook
(the server, the index, the tool/resource/prompt loading) is identical
either way, since none of it depends on Groq.